# W09 — Assignment único semanal (Limpieza avanzada + Quality Gates)

## Setup

In [3]:
from pathlib import Path
import duckdb
import os
#os.chdir("..")
print("cwd:", os.getcwd())

PROJECT_ROOT = Path(".").resolve()
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Missing {DB_PATH}. Run W06 pipeline first.")

con = duckdb.connect(str(DB_PATH))

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'","''") + "'"

if not RAW_CSV.exists():
    raise FileNotFoundError(f"Missing {RAW_CSV}")

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

cwd: c:\Users\nancy\Documents\Ingenieria_datos


## Parte A — Limpieza avanzada

In [4]:
# TODO 1: method_synonyms(raw_norm, canonical)

# TODO: CREATE TABLE + INSERTs

con.execute("DROP TABLE IF EXISTS method_synonyms")

con.execute("CREATE TABLE method_synonyms(raw_norm VARCHAR, canonical VARCHAR)")

con.execute("""
INSERT INTO method_synonyms VALUES
    ('transit',                        'transit'),
    ('radial velocity',                'radial_velocity'),
    ('imaging',                        'imaging'),
    ('microlensing',                   'microlensing'),
    ('transit timing variations',      'transit_timing_variations'),
    ('eclipse timing variations',      'eclipse_timing_variations'),
    ('astrometry',                     'astrometry'),
    ('orbital brightness modulation',  'orbital_brightness_modulation'),
    ('pulsar timing',                  'pulsar_timing'),
    ('pulsation timing variations',    'pulsation_timing_variations'),
    ('disk kinematics',                'disk_kinematics')
""")

con.sql("SELECT * FROM method_synonyms").show()

┌───────────────────────────────┬───────────────────────────────┐
│           raw_norm            │           canonical           │
│            varchar            │            varchar            │
├───────────────────────────────┼───────────────────────────────┤
│ transit                       │ transit                       │
│ radial velocity               │ radial_velocity               │
│ imaging                       │ imaging                       │
│ microlensing                  │ microlensing                  │
│ transit timing variations     │ transit_timing_variations     │
│ eclipse timing variations     │ eclipse_timing_variations     │
│ astrometry                    │ astrometry                    │
│ orbital brightness modulation │ orbital_brightness_modulation │
│ pulsar timing                 │ pulsar_timing                 │
│ pulsation timing variations   │ pulsation_timing_variations   │
│ disk kinematics               │ disk_kinematics               │
├─────────

In [5]:
# TODO 2: silver_planet_v3
# Debe incluir:
# - hostname_canon
# - discoverymethod_canon (synonyms + COALESCE fallback)
# - disc_year_int = TRY_CAST(disc_year AS INTEGER)
# - disc_year_bad flag

con.execute("DROP TABLE IF EXISTS silver_planet_v3")

# TODO: CREATE TABLE AS SELECT ...

con.execute("""
CREATE TABLE silver_planet_v3 AS
SELECT
    pl_name,
    hostname,
    LOWER(TRIM(hostname))                          AS hostname_canon,
    discoverymethod,
    LOWER(TRIM(discoverymethod))                   AS discoverymethod_norm,
    COALESCE(
        s.canonical,
        LOWER(TRIM(discoverymethod))
    )                                              AS discoverymethod_canon,
    disc_year,
    TRY_CAST(disc_year AS INTEGER)                 AS disc_year_int,
    CASE
        WHEN TRY_CAST(disc_year AS INTEGER) IS NULL THEN true
        WHEN TRY_CAST(disc_year AS INTEGER) < 1980  THEN true
        WHEN TRY_CAST(disc_year AS INTEGER) > 2026  THEN true
        ELSE false
    END                                            AS disc_year_bad,
    pl_orbper,
    pl_rade,
    pl_bmasse,
    pl_eqt,
    sy_dist,
    ra,
    dec
FROM raw_ps
LEFT JOIN method_synonyms s
    ON LOWER(TRIM(raw_ps.discoverymethod)) = s.raw_norm
WHERE pl_name  IS NOT NULL
  AND hostname IS NOT NULL
  AND (pl_rade   IS NULL OR (pl_rade > 0 AND pl_rade <= 30))
  AND (pl_bmasse IS NULL OR pl_bmasse > 0)
""")

con.sql("SELECT COUNT(*) AS n_rows FROM silver_planet_v3").show()
con.sql("SELECT COUNT(*) AS disc_year_bad FROM silver_planet_v3 WHERE disc_year_bad").show()

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│   6101 │
└────────┘

┌───────────────┐
│ disc_year_bad │
│     int64     │
├───────────────┤
│             1 │
└───────────────┘



## Parte B — Quality gates

In [6]:
# TODO 3: quality_events + 4 checks
# Crea tabla:
# ts_utc, check_name, status, metric_value, details

from datetime import datetime, timezone

con.execute("DROP TABLE IF EXISTS quality_events")

con.execute("""
CREATE TABLE quality_events(
    ts_utc       TIMESTAMPTZ,
    check_name   VARCHAR,
    status       VARCHAR,
    metric_value DOUBLE,
    details      VARCHAR
)
""")

ts = datetime.now(timezone.utc).isoformat()

# Check 1: filas totales en silver_planet_v3
n_rows = con.execute("SELECT COUNT(*) FROM silver_planet_v3").fetchone()[0]
status_rows = 'PASS' if n_rows >= 6000 else 'FAIL'
con.execute("INSERT INTO quality_events VALUES (?, 'row_count_silver_v3', ?, ?, ?)",
    [ts, status_rows, float(n_rows),
     f"Expected >= 6000, got {n_rows}"])

# Check 2: hosts nulos en hostname_canon
null_hosts = con.execute(
    "SELECT COUNT(*) FROM silver_planet_v3 WHERE hostname_canon IS NULL"
).fetchone()[0]
status_hosts = 'PASS' if null_hosts == 0 else 'FAIL'
con.execute("INSERT INTO quality_events VALUES (?, 'null_hostname_canon', ?, ?, ?)",
    [ts, status_hosts, float(null_hosts),
     f"Expected 0 null hosts, got {null_hosts}"])

# Check 3: años con disc_year_bad
bad_years = con.execute(
    "SELECT COUNT(*) FROM silver_planet_v3 WHERE disc_year_bad"
).fetchone()[0]
status_years = 'PASS' if bad_years == 0 else 'WARN'
con.execute("INSERT INTO quality_events VALUES (?, 'disc_year_bad_count', ?, ?, ?)",
    [ts, status_years, float(bad_years),
     f"Rows with bad disc_year: {bad_years}"])

# Check 4: métodos canónicos únicos
n_methods = con.execute(
    "SELECT COUNT(DISTINCT discoverymethod_canon) FROM silver_planet_v3 WHERE discoverymethod_canon IS NOT NULL"
).fetchone()[0]
status_methods = 'PASS' if n_methods >= 6 else 'FAIL'
con.execute("INSERT INTO quality_events VALUES (?, 'canonical_method_count', ?, ?, ?)",
    [ts, status_methods, float(n_methods),
     f"Expected >= 6 canonical methods, got {n_methods}"])

con.sql("SELECT check_name, status, metric_value FROM quality_events ORDER BY check_name").show()

┌────────────────────────┬─────────┬──────────────┐
│       check_name       │ status  │ metric_value │
│        varchar         │ varchar │    double    │
├────────────────────────┼─────────┼──────────────┤
│ canonical_method_count │ PASS    │         11.0 │
│ disc_year_bad_count    │ WARN    │          1.0 │
│ null_hostname_canon    │ PASS    │          0.0 │
│ row_count_silver_v3    │ PASS    │       6101.0 │
└────────────────────────┴─────────┴──────────────┘



## Entregable único semanal (W09)

Entrega:
- `assignments/W09_assignment_student.ipynb` ejecutado
- `docs/w09_report.md` (usar template)
- `docs/w09_quality.md` (usar template)
- 1 entrada en `docs/decisions_log.md` (usar template)